# 02 - Preprocessing and leakage-controlled splitting

Turns the seven downloaded corpora into one canonical table with a shared
symptom label space, then assigns train/val/test so that no post and no Reddit
author can appear on both sides of a split.

## Why both steps are necessary

The datasets cannot be used as they are. Measured on the data you downloaded:

| Finding | Consequence |
|---|---|
| **DepSeverity is a re-annotation of Dreaddit** - the same 3,526 posts | Dreaddit cannot be an out-of-domain test set, because it *is* the severity training set |
| **LoST and IRF share 3,089 posts** | Splitting them separately puts the same post in train and test |
| **IMHI re-packages the IRF and Dreaddit posts** behind a `"Post: "` prefix | Exact-match deduplication misses it entirely |
| **23.3% of Reddit authors (192,893 of 826,961) appear in more than one file** | Random splitting puts the same person in train and test |
| **85,659 Reddit posts are duplicated across period files** | Inflates the corpus and leaks across the temporal split |

Of 46,542 rows in the labelled corpora, only **19,762 are distinct posts**. Those
overlaps are why splitting has to happen once, globally, over unique posts.

## What this notebook produces

- `labelled.parquet` - 19,762 posts with symptom, severity, risk, stress, span and explanation labels
- `weak_pool.parquet` - 1,017,959 Reddit posts for LLM weak labelling (novelty N3), author-disjoint
- `label_coverage.csv` - which of the 22 symptoms have gold labels and which need weak labels
- `manifest.json` - counts, settings and the leakage audit

Runtime is about 8 minutes and peaks near 1.7 GB of RAM, so a free Colab runtime
is enough.

## 1. Mount Drive and set paths

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/hari/SymptoGraph-LLM/data'
DRIVE_OUT  = '/content/drive/MyDrive/hari/SymptoGraph-LLM/processed'

# Reading 108 gzip files through the Drive mount is slow, so work on local disk
# and copy the results back at the end.
LOCAL_DATA = '/content/data'
LOCAL_OUT  = '/content/processed'

# True runs on 300 posts per Reddit file (about 1 minute) to check the pipeline
# end to end before committing to the full run.
QUICK_TEST = False

os.environ.update(DRIVE_DATA=DRIVE_DATA, DRIVE_OUT=DRIVE_OUT,
                  LOCAL_DATA=LOCAL_DATA, LOCAL_OUT=LOCAL_OUT)
assert os.path.isdir(DRIVE_DATA), f'not found: {DRIVE_DATA}'
print('data in Drive:', sorted(os.listdir(DRIVE_DATA)))

## 2. Get the code

A checkout left over from an earlier session will be missing the preprocessing
module, so this fails loudly rather than silently using stale code.

In [ ]:
REPO = 'https://github.com/sid-2k6/SymptoGraph-LLM.git'
CHECKOUT = '/content/SymptoGraph-LLM'

if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q --branch main $REPO $CHECKOUT

need = ['src/symptograph/preprocess.py', 'configs/symptom_ontology.json']
missing = [f for f in need if not os.path.isfile(f'{CHECKOUT}/{f}')]
assert not missing, f'stale checkout, missing {missing} - run: !rm -rf {CHECKOUT}'

import sys
sys.path.insert(0, f'{CHECKOUT}/src')
!git -C $CHECKOUT log --oneline -1

## 3. Copy the data to local disk

About 0.6 GB. Skipped automatically if it is already there.

In [ ]:
!mkdir -p $LOCAL_DATA
!rsync -ah --info=progress2 "$DRIVE_DATA/" "$LOCAL_DATA/"
!python $CHECKOUT/scripts/verify_data.py --data "$LOCAL_DATA"

## 4. The unified symptom ontology

The concept bottleneck (novelty N1) predicts these 22 symptoms. Nine come from
the PHQ-9, seven from the GAD-7, and six are additional concepts that the open
datasets actually annotate. Each disorder and severity prediction is forced
through this layer, which is what makes the model interpretable by construction.

In [ ]:
import pandas as pd
from symptograph import preprocess as P

onto = P.Ontology.load(f'{CHECKOUT}/configs/symptom_ontology.json')
print(f'{len(onto.symptoms)} symptoms\n')
display(pd.DataFrame(onto.meta)[['name', 'instrument', 'item', 'definition']])
print('DepressionEmo label -> unified symptom:')
for k, v in onto.label_map['depressionemo'].items():
    print(f'  {k:28s} -> {v}')

## 5. Run the pipeline

Labelled corpora are processed in memory; the 1.1M-post Reddit pool is streamed
file by file and written incrementally. Streaming is what keeps peak memory near
1.7 GB instead of the 16 GB a single in-memory pass needs.

The run asserts that no post and no author crosses a split, and raises if that
ever fails.

In [ ]:
import time

t0 = time.time()
df, paths, stats, report = P.run(
    data_dir=LOCAL_DATA,
    out_dir=LOCAL_OUT,
    include_weak_pool=True,
    stream_weak=True,                 # keeps memory inside a free runtime
    rmhd_max_per_file=300 if QUICK_TEST else None,
    verbose=True,
)
print(f'\nfinished in {time.time() - t0:.0f}s')

## 6. What came out

In [ ]:
import json

print('Labelled corpora')
for k in ['loaded_rows', 'dropped_short_or_empty', 'rows_before_merge',
          'rows_needing_merge', 'unique_posts', 'duplicate_rows_merged']:
    print(f'  {k:28s} {stats[k]:>10,}')

if 'weak_pool' in stats:
    print('\nReddit weak pool')
    for k, v in stats['weak_pool'].items():
        if k != 'split_sizes':
            print(f'  {k:28s} {v:>10,}')

print('\nSplit sizes')
print('  labelled:', report['split_sizes'])
if 'weak_pool' in stats:
    print('  weak pool:', stats['weak_pool']['split_sizes'])

### Split strategy per post

Precedence: external test sets stay test-only; the severity and risk pools are
stratified on their own label; otherwise a published split is honoured (and when
two corpora disagree the strictest wins, so we never train on another
benchmark's test post); Reddit falls back to author-disjoint hashing.

In [ ]:
print(df.split_strategy.value_counts().to_string())
print()
display(df.groupby([df['pool'].astype(str), 'split']).size().unstack(fill_value=0))

### Symptom label coverage

Eleven of the 22 symptoms have gold labels from the open corpora. The remaining
eleven, mostly GAD-7 anxiety items, have no open gold data at all - which is
exactly the gap that LLM weak supervision (novelty N3) plus a small in-house
anxiety gold set is designed to fill. Reporting symptom F1 separately for these
two groups is what makes the N3 claim measurable.

In [ ]:
cov = P.label_coverage(df, onto)
display(cov)
print(cov.supervision.value_counts().to_string())

### Training targets available in each split

In [ ]:
targets = pd.DataFrame({
    'symptom_gold': df[(df[onto.columns] >= 0).any(axis=1)].groupby('split').size(),
    'severity_depression': df[df.severity_depression.notna()].groupby('split').size(),
    'severity_risk': df[df.severity_risk.notna()].groupby('split').size(),
    'stress': df[df.stress.notna()].groupby('split').size(),
    'evidence_spans': df[df.spans.notna()].groupby('split').size(),
    'imhi_explanations': df[df.imhi_response.notna()].groupby('split').size(),
}).fillna(0).astype(int)
display(targets)

sev = df[df.severity_depression.notna()]
print('Depression severity by split (0 minimum .. 3 severe)')
display(pd.crosstab(sev.split, sev.severity_depression))
risk = df[df.severity_risk.notna()]
print('C-SSRS risk by split (0 supportive .. 4 attempt)')
display(pd.crosstab(risk.split, risk.severity_risk))

## 7. Independent leakage audit

Re-reads the written files rather than trusting the in-memory result. All four
counts must be zero.

In [ ]:
lab = pd.read_parquet(f'{LOCAL_OUT}/labelled.parquet',
                      columns=['post_id', 'split', 'author_hash'])
checks = {
    'duplicate post_id in labelled': int(lab.post_id.duplicated().sum()),
}
weak_path = f'{LOCAL_OUT}/weak_pool.parquet'
if os.path.exists(weak_path):
    weak = pd.read_parquet(weak_path, columns=['post_id', 'author_hash', 'split', 'period'])
    spans = weak[weak.author_hash.notna()].groupby('author_hash')['split'].nunique()
    checks.update({
        'duplicate post_id in weak pool': int(weak.post_id.duplicated().sum()),
        'posts shared by both files': len(set(weak.post_id) & set(lab.post_id)),
        'authors spanning splits': int((spans > 1).sum()),
    })

for k, v in checks.items():
    print(f'{k:34s} {v:>8,}  {"ok" if v == 0 else "LEAK"}')
assert all(v == 0 for v in checks.values()), 'leakage detected'
print('\nNo leakage.')

if os.path.exists(weak_path):
    print(f'\n{len(spans):,} distinct authors, each confined to one split')
    print('\nPeriod by split (supports the pre/mid-COVID temporal test):')
    display(pd.crosstab(weak.period, weak.split))

## 8. A processed example

In [ ]:
row = df[(df.spans.notna()) & (df[onto.col('thwarted_belongingness')] == 1)].iloc[0]
print('post_id  :', row.post_id)
print('source   :', row.source)
print('pool     :', row.pool, '| split:', row.split, f'({row.split_strategy})')
print('text     :', row.text[:300], '...')
present = [s for s in onto.symptoms if row[onto.col(s)] == 1]
absent = [s for s in onto.symptoms if row[onto.col(s)] == 0]
print('\nsymptoms present :', present)
print('symptoms absent  :', absent)
print('not annotated    :', len(onto.symptoms) - len(present) - len(absent), 'symptoms (-1)')
print('\nevidence spans   :', row.spans)

## 9. Copy the results to Drive

In [ ]:
!mkdir -p "$DRIVE_OUT"
!rsync -ah --info=progress2 "$LOCAL_OUT/" "$DRIVE_OUT/"
!ls -lh "$DRIVE_OUT"
print()
print(json.dumps(json.load(open(f'{DRIVE_OUT}/manifest.json'))['counts'], indent=2))

## Notes

**Label masking matters.** Symptom columns use `1` present, `0` absent and `-1`
not annotated by that source. Only non-negative entries should contribute to the
concept loss - treating `-1` as a negative would teach the model that
unannotated symptoms are absent.

**Reading the outputs later:**

```python
import pandas as pd
base = '/content/drive/MyDrive/hari/SymptoGraph-LLM/processed'
lab = pd.read_parquet(f'{base}/labelled.parquet')
train = lab[lab.split == 'train']
```

The weak pool is 427 MB, so filter it by column or row group rather than loading
it whole when you only need part of it.

**Design change this notebook forced.** Dreaddit was originally planned as the
out-of-domain test set, but it is the same corpus as DepSeverity. Its stress
labels are now an auxiliary target on shared posts, and external evaluation uses
the IMHI subsets that genuinely do not overlap anything else: `swmh`, `CAMS`,
`t-sid`, `loneliness` and `DR`. Those are held as test-only.

**Next:** notebook 03 builds the symptom-disorder comorbidity graph (novelty N2)
from training-split co-occurrence only, and runs LLM weak labelling (N3) over the
weak pool.